In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
Implement Grouped Query Attention (GQA), the attention mechanism used in modern large language
models such as LLaMA-3, Mistral, and Gemma. GQA reduces the KV-cache memory footprint during
inference by sharing key and value heads across groups of query heads. Given query tensor
<code>Q</code> with <code>num_q_heads</code> heads and key/value tensors <code>K</code>,
<code>V</code> each with <code>num_kv_heads</code> heads, compute scaled dot-product attention
where every group of <code>num_q_heads / num_kv_heads</code> consecutive query heads attends to
the same key and value head. All tensors use <code>float32</code>.
</p>

<svg width="700" height="260" viewBox="0 0 700 260" xmlns="http://www.w3.org/2000/svg" style="display:block; margin:20px auto;">
  <rect width="700" height="260" fill="#222" rx="10"/>
  <!-- Title -->
  <text x="350" y="28" fill="#ccc" font-family="monospace" font-size="14" text-anchor="middle">Grouped Query Attention (num_q_heads=4, num_kv_heads=2, groups=2)</text>

  <!-- Q heads -->
  <text x="80" y="60" fill="#aaa" font-family="monospace" font-size="12" text-anchor="middle">Q heads</text>
  <rect x="20" y="70" width="60" height="36" fill="#2563eb" rx="4"/>
  <text x="50" y="93" fill="#fff" font-family="monospace" font-size="12" text-anchor="middle">Q[0]</text>
  <rect x="100" y="70" width="60" height="36" fill="#2563eb" rx="4"/>
  <text x="130" y="93" fill="#fff" font-family="monospace" font-size="12" text-anchor="middle">Q[1]</text>
  <rect x="180" y="70" width="60" height="36" fill="#7c3aed" rx="4"/>
  <text x="210" y="93" fill="#fff" font-family="monospace" font-size="12" text-anchor="middle">Q[2]</text>
  <rect x="260" y="70" width="60" height="36" fill="#7c3aed" rx="4"/>
  <text x="290" y="93" fill="#fff" font-family="monospace" font-size="12" text-anchor="middle">Q[3]</text>

  <!-- KV heads -->
  <text x="80" y="175" fill="#aaa" font-family="monospace" font-size="12" text-anchor="middle">KV heads</text>
  <rect x="20" y="185" width="120" height="36" fill="#1d4ed8" rx="4"/>
  <text x="80" y="208" fill="#fff" font-family="monospace" font-size="12" text-anchor="middle">K[0], V[0]</text>
  <rect x="180" y="185" width="120" height="36" fill="#5b21b6" rx="4"/>
  <text x="240" y="208" fill="#fff" font-family="monospace" font-size="12" text-anchor="middle">K[1], V[1]</text>

  <!-- Arrows group 0 -->
  <line x1="50" y1="106" x2="70" y2="185" stroke="#60a5fa" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="130" y1="106" x2="90" y2="185" stroke="#60a5fa" stroke-width="1.5" marker-end="url(#arr)"/>

  <!-- Arrows group 1 -->
  <line x1="210" y1="106" x2="230" y2="185" stroke="#c4b5fd" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="290" y1="106" x2="250" y2="185" stroke="#c4b5fd" stroke-width="1.5" marker-end="url(#arr)"/>

  <!-- Output boxes -->
  <text x="80" y="245" fill="#aaa" font-family="monospace" font-size="11" text-anchor="middle">group 0</text>
  <text x="240" y="245" fill="#aaa" font-family="monospace" font-size="11" text-anchor="middle">group 1</text>

  <!-- bracket labels -->
  <text x="430" y="88" fill="#60a5fa" font-family="monospace" font-size="12">Q[0], Q[1] attend to K[0], V[0]</text>
  <text x="430" y="112" fill="#c4b5fd" font-family="monospace" font-size="12">Q[2], Q[3] attend to K[1], V[1]</text>
  <text x="430" y="150" fill="#4ade80" font-family="monospace" font-size="12">scale = 1 / sqrt(head_dim)</text>
  <text x="430" y="174" fill="#4ade80" font-family="monospace" font-size="12">scores = Q @ K^T * scale</text>
  <text x="430" y="198" fill="#4ade80" font-family="monospace" font-size="12">weights = softmax(scores)</text>
  <text x="430" y="222" fill="#4ade80" font-family="monospace" font-size="12">output = weights @ V</text>

  <defs>
    <marker id="arr" markerWidth="6" markerHeight="6" refX="3" refY="3" orient="auto">
      <path d="M0,0 L0,6 L6,3 z" fill="#888"/>
    </marker>
  </defs>
</svg>

<h2>Implementation Requirements</h2>
<ul>
  <li>Implement the function <code>solve(Q, K, V, output, num_q_heads, num_kv_heads, seq_len, head_dim)</code>.</li>
  <li>Do not change the function signature or use external libraries beyond the standard GPU frameworks.</li>
  <li>Write the result into the provided <code>output</code> buffer.</li>
  <li><code>num_q_heads</code> is always divisible by <code>num_kv_heads</code>.</li>
  <li>Use scaled dot-product attention with scale factor <code>1 / sqrt(head_dim)</code> and a softmax over the key dimension.</li>
</ul>

<h2>Example</h2>
<p>
  With <code>num_q_heads</code> = 4, <code>num_kv_heads</code> = 2 (groups of 2), <code>seq_len</code> = 3,
  <code>head_dim</code> = 4:
</p>
<p>
  <strong>Input:</strong><br>
  $Q_0$ (3&times;4):
  $$
  \begin{bmatrix}
  1 & 0 & 0 & 1 \\
  0 & 1 & 1 & 0 \\
  1 & 1 & 0 & 0
  \end{bmatrix}
  $$
  $Q_1$ (3&times;4):
  $$
  \begin{bmatrix}
  0 & 1 & 0 & 1 \\
  1 & 0 & 1 & 0 \\
  0 & 0 & 1 & 1
  \end{bmatrix}
  $$
  $Q_2$ (3&times;4):
  $$
  \begin{bmatrix}
  -1 & 0 & 0.5 & 0 \\
  0 & -1 & 0 & 0.5 \\
  0.5 & 0 & -1 & 0
  \end{bmatrix}
  $$
  $Q_3$ (3&times;4):
  $$
  \begin{bmatrix}
  0 & 0.5 & 0 & -1 \\
  0.5 & 0 & 0 & -1 \\
  0 & 0 & 0.5 & 0.5
  \end{bmatrix}
  $$
  $K_0$ (3&times;4):
  $$
  \begin{bmatrix}
  1 & 0 & 1 & 0 \\
  0 & 1 & 0 & 1 \\
  1 & 1 & 1 & 1
  \end{bmatrix}
  $$
  $K_1$ (3&times;4):
  $$
  \begin{bmatrix}
  0 & 1 & 0 & -1 \\
  -1 & 0 & 1 & 0 \\
  0 & -1 & 0 & 1
  \end{bmatrix}
  $$
  $V_0$ (3&times;4):
  $$
  \begin{bmatrix}
  1 & 2 & 3 & 4 \\
  5 & 6 & 7 & 8 \\
  9 & 10 & 11 & 12
  \end{bmatrix}
  $$
  $V_1$ (3&times;4):
  $$
  \begin{bmatrix}
  -1 & -2 & -3 & -4 \\
  2 & 3 & 4 & 5 \\
  6 & 7 & 8 & 9
  \end{bmatrix}
  $$
  Groups: $Q_0, Q_1 \to K_0, V_0$; \quad $Q_2, Q_3 \to K_1, V_1$
</p>
<p>
  <strong>Output</strong> (values rounded to 2 decimal places):<br>
  $\text{output}_0$ (3&times;4):
  $$
  \begin{bmatrix}
  5.71 & 6.71 & 7.71 & 8.71 \\
  5.71 & 6.71 & 7.71 & 8.71 \\
  5.71 & 6.71 & 7.71 & 8.71
  \end{bmatrix}
  $$
  $\text{output}_1$ (3&times;4):
  $$
  \begin{bmatrix}
  6.07 & 7.07 & 8.07 & 9.07 \\
  5.00 & 6.00 & 7.00 & 8.00 \\
  5.71 & 6.71 & 7.71 & 8.71
  \end{bmatrix}
  $$
  $\text{output}_2$ (3&times;4):
  $$
  \begin{bmatrix}
  2.24 & 2.76 & 3.27 & 3.79 \\
  3.96 & 4.70 & 5.44 & 6.17 \\
  2.40 & 2.60 & 2.79 & 2.98
  \end{bmatrix}
  $$
  $\text{output}_3$ (3&times;4):
  $$
  \begin{bmatrix}
  0.76 & 0.58 & 0.40 & 0.22 \\
  1.17 & 1.08 & 1.00 & 0.91 \\
  2.84 & 3.37 & 3.91 & 4.44
  \end{bmatrix}
  $$
</p>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>num_kv_heads</code> &le; <code>num_q_heads</code> &le; 64</li>
  <li><code>num_q_heads</code> is divisible by <code>num_kv_heads</code></li>
  <li>1 &le; <code>seq_len</code> &le; 4,096</li>
  <li>8 &le; <code>head_dim</code> &le; 256; <code>head_dim</code> is a multiple of 8</li>
  <li>All tensor values are <code>float32</code></li>
  <li>Performance is measured with <code>num_q_heads</code> = 32, <code>num_kv_heads</code> = 8, <code>seq_len</code> = 1,024, <code>head_dim</code> = 128</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// Q, K, V, output are device pointers
extern "C" void solve(const float* Q, const float* K, const float* V, float* output,
                      int num_q_heads, int num_kv_heads, int seq_len, int head_dim) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# Q, K, V, output are tensors on the GPU
@cute.jit
def solve(
    Q: cute.Tensor,
    K: cute.Tensor,
    V: cute.Tensor,
    output: cute.Tensor,
    num_q_heads: cute.Int32,
    num_kv_heads: cute.Int32,
    seq_len: cute.Int32,
    head_dim: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# Q, K, V are tensors on GPU
@jax.jit
def solve(
    Q: jax.Array,
    K: jax.Array,
    V: jax.Array,
    num_q_heads: int,
    num_kv_heads: int,
    seq_len: int,
    head_dim: int,
) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.memory import UnsafePointer


# Q, K, V, output are device pointers
@export
def solve(
    Q: UnsafePointer[Float32, MutExternalOrigin],
    K: UnsafePointer[Float32, MutExternalOrigin],
    V: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    num_q_heads: Int32,
    num_kv_heads: Int32,
    seq_len: Int32,
    head_dim: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# Q, K, V, output are tensors on the GPU
def solve(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    output: torch.Tensor,
    num_q_heads: int,
    num_kv_heads: int,
    seq_len: int,
    head_dim: int,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# Q, K, V, output are tensors on the GPU
def solve(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    output: torch.Tensor,
    num_q_heads: int,
    num_kv_heads: int,
    seq_len: int,
    head_dim: int,
):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
import math
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Grouped Query Attention",
            atol=1e-04,
            rtol=1e-04,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(
        self,
        Q: torch.Tensor,
        K: torch.Tensor,
        V: torch.Tensor,
        output: torch.Tensor,
        num_q_heads: int,
        num_kv_heads: int,
        seq_len: int,
        head_dim: int,
    ):
        assert Q.shape == (num_q_heads, seq_len, head_dim)
        assert K.shape == (num_kv_heads, seq_len, head_dim)
        assert V.shape == (num_kv_heads, seq_len, head_dim)
        assert output.shape == (num_q_heads, seq_len, head_dim)
        assert Q.dtype == K.dtype == V.dtype == output.dtype == torch.float32
        assert Q.device.type == "cuda"
        assert K.device.type == "cuda"
        assert V.device.type == "cuda"
        assert output.device.type == "cuda"
        assert num_q_heads % num_kv_heads == 0

        num_groups = num_q_heads // num_kv_heads
        scale = 1.0 / math.sqrt(head_dim)

        # Expand K, V from (num_kv_heads, seq_len, head_dim)
        # to (num_q_heads, seq_len, head_dim) by repeating each KV head num_groups times
        K_expanded = K.repeat_interleave(num_groups, dim=0)
        V_expanded = V.repeat_interleave(num_groups, dim=0)

        # Scaled dot-product attention: (num_q_heads, seq_len, seq_len)
        scores = torch.bmm(Q, K_expanded.transpose(1, 2)) * scale

        # Softmax over the key dimension
        attn_weights = torch.softmax(scores, dim=-1)

        # Weighted sum of values: (num_q_heads, seq_len, head_dim)
        output.copy_(torch.bmm(attn_weights, V_expanded))

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "Q": (ctypes.POINTER(ctypes.c_float), "in"),
            "K": (ctypes.POINTER(ctypes.c_float), "in"),
            "V": (ctypes.POINTER(ctypes.c_float), "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "num_q_heads": (ctypes.c_int, "in"),
            "num_kv_heads": (ctypes.c_int, "in"),
            "seq_len": (ctypes.c_int, "in"),
            "head_dim": (ctypes.c_int, "in"),
        }

    def _make_test_case(self, num_q_heads, num_kv_heads, seq_len, head_dim, zero_inputs=False):
        dtype = torch.float32
        device = "cuda"
        if zero_inputs:
            Q = torch.zeros(num_q_heads, seq_len, head_dim, device=device, dtype=dtype)
            K = torch.zeros(num_kv_heads, seq_len, head_dim, device=device, dtype=dtype)
            V = torch.zeros(num_kv_heads, seq_len, head_dim, device=device, dtype=dtype)
        else:
            Q = torch.randn(num_q_heads, seq_len, head_dim, device=device, dtype=dtype)
            K = torch.randn(num_kv_heads, seq_len, head_dim, device=device, dtype=dtype)
            V = torch.randn(num_kv_heads, seq_len, head_dim, device=device, dtype=dtype)
        output = torch.zeros(num_q_heads, seq_len, head_dim, device=device, dtype=dtype)
        return {
            "Q": Q,
            "K": K,
            "V": V,
            "output": output,
            "num_q_heads": num_q_heads,
            "num_kv_heads": num_kv_heads,
            "seq_len": seq_len,
            "head_dim": head_dim,
        }

    def generate_example_test(self) -> Dict[str, Any]:
        torch.manual_seed(0)
        dtype = torch.float32
        device = "cuda"
        num_q_heads = 4
        num_kv_heads = 2
        seq_len = 3
        head_dim = 4

        Q = torch.tensor(
            [
                [[1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 1.0, 0.0], [1.0, 1.0, 0.0, 0.0]],
                [[0.0, 1.0, 0.0, 1.0], [1.0, 0.0, 1.0, 0.0], [0.0, 0.0, 1.0, 1.0]],
                [[-1.0, 0.0, 0.5, 0.0], [0.0, -1.0, 0.0, 0.5], [0.5, 0.0, -1.0, 0.0]],
                [[0.0, 0.5, 0.0, -1.0], [0.5, 0.0, 0.0, -1.0], [0.0, 0.0, 0.5, 0.5]],
            ],
            device=device,
            dtype=dtype,
        )
        K = torch.tensor(
            [
                [[1.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 1.0], [1.0, 1.0, 1.0, 1.0]],
                [[0.0, 1.0, 0.0, -1.0], [-1.0, 0.0, 1.0, 0.0], [0.0, -1.0, 0.0, 1.0]],
            ],
            device=device,
            dtype=dtype,
        )
        V = torch.tensor(
            [
                [[1.0, 2.0, 3.0, 4.0], [5.0, 6.0, 7.0, 8.0], [9.0, 10.0, 11.0, 12.0]],
                [[-1.0, -2.0, -3.0, -4.0], [2.0, 3.0, 4.0, 5.0], [6.0, 7.0, 8.0, 9.0]],
            ],
            device=device,
            dtype=dtype,
        )
        output = torch.zeros(num_q_heads, seq_len, head_dim, device=device, dtype=dtype)
        return {
            "Q": Q,
            "K": K,
            "V": V,
            "output": output,
            "num_q_heads": num_q_heads,
            "num_kv_heads": num_kv_heads,
            "seq_len": seq_len,
            "head_dim": head_dim,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        torch.manual_seed(42)
        tests = []

        # Edge case: MQA (num_kv_heads=1), single token
        tests.append(self._make_test_case(4, 1, 1, 8))

        # Edge case: GQA with groups=2, tiny seq
        tests.append(self._make_test_case(2, 1, 2, 4))

        # Zero inputs
        tests.append(self._make_test_case(4, 2, 4, 8, zero_inputs=True))

        # Power-of-2: groups=4 (LLaMA-3 style ratio)
        tests.append(self._make_test_case(8, 2, 16, 32))

        # Power-of-2: seq_len=32, head_dim=64
        tests.append(self._make_test_case(4, 2, 32, 64))

        # Non-power-of-2 seq_len
        tests.append(self._make_test_case(4, 2, 30, 32))

        # Non-power-of-2 seq_len, different grouping
        tests.append(self._make_test_case(6, 3, 100, 32))

        # GQA groups=8 (Mistral style), seq_len=255
        tests.append(self._make_test_case(8, 1, 255, 64))

        # MHA equivalent (num_q_heads == num_kv_heads)
        tests.append(self._make_test_case(8, 8, 64, 32))

        # Realistic small inference batch
        tests.append(self._make_test_case(8, 2, 128, 64))

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        torch.manual_seed(0)
        # LLaMA-3 8B style: 32 Q heads, 8 KV heads, head_dim=128
        return self._make_test_case(32, 8, 1024, 128)


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
